# PreToolUse Hook Output Shapes

**Purpose:** Documents what was learned about PreToolUse response shapes, then **validates those findings live** by running `claude -p` with a configurable PreToolUse hook. Each response shape is tested and the agent's behavioral response is observed.

Source: `research/plugins/pre-tool-use-test/` — configurable PreToolUse response plugin

## Background

PreToolUse hooks run **before** a tool executes. They can block the tool call by returning a specific response shape.

The hook receives a JSON payload on stdin and must print a JSON response to stdout. The response shape determines whether the tool is blocked:

- **`continue: true`** — allow the tool to proceed (default pass-through)
- **`deny` via `hookSpecificOutput`** — block the tool call and surface a reason
- **`error`** — signals a hook failure (NOT a block — agents treat this as transient)

The spike explored which response shapes actually prevent tool execution and which don't.

## Tested Response Shapes

The shell script tested three response shapes. Here they are as Python dicts for clarity:

In [ ]:
import json

# Shape 1: What the spike originally tested — systemMessage + hookSpecificOutput together
# The experiment asked: does systemMessage go to the agent, the user, or both?
shape_tested = {
    "systemMessage": "USER-ONLY? This message tests if systemMessage is visible to the agent.",
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "Blocked for testing."
    }
}

# Shape 2: Error response — what happens when a hook script exits with an error
# Agents interpret this as a TRANSIENT FAILURE and retry with variations
shape_error = {
    "error": "Hook script failed unexpectedly"
}

# Shape 3: The correct minimal deny shape — blocks the tool call cleanly
shape_deny = {
    "hookSpecificOutput": {
        "hookEventName": "PreToolUse",
        "permissionDecision": "deny",
        "permissionDecisionReason": "This tool call is not permitted in this context."
    }
}

print("Shape 1 (tested — systemMessage + deny):")
print(json.dumps(shape_tested, indent=2))
print()
print("Shape 2 (error — triggers retry loops):")
print(json.dumps(shape_error, indent=2))
print()
print("Shape 3 (correct — minimal deny):")
print(json.dumps(shape_deny, indent=2))

## Live Behavioral Tests

Each test writes a response shape to `/tmp/claude/pre-tool-use-response.json`, runs `claude -p` with the `pre-tool-use-test` plugin, and observes what the agent does.

In [ ]:
# Setup — define plugin dir, response file, test helper
import json
from pathlib import Path
from lib import run_claude

RESPONSE_FILE = Path("/tmp/claude/pre-tool-use-response.json")
PLUGIN_DIR = Path("research/plugins/pre-tool-use-test").resolve()
TOOL_PROMPT = "Use Bash to run: echo hello"

def run_test(response_shape: dict) -> dict:
    """Write a PreToolUse response shape, run claude -p, return parsed JSON output."""
    RESPONSE_FILE.parent.mkdir(parents=True, exist_ok=True)
    RESPONSE_FILE.write_text(json.dumps(response_shape))
    cr = run_claude(TOOL_PROMPT, plugin_dir=PLUGIN_DIR)
    RESPONSE_FILE.unlink(missing_ok=True)
    return cr.output

print(f"Plugin dir: {PLUGIN_DIR}")
print(f"Response file: {RESPONSE_FILE}")
print("run_test() ready.")

In [ ]:
# Test: deny shape — should block the tool call cleanly
print("=" * 60)
print("TEST: deny via hookSpecificOutput.permissionDecision")
print("=" * 60)

output = run_test(shape_deny)

# Display the result
print(f"\nResult type: {type(output).__name__}")
if "result" in output:
    text = output["result"].get("content", [{}])[0].get("text", "(no text)")
    print(f"\nAgent response:\n{text[:500]}")
elif "error" in output:
    print(f"\nError: {output['error'][:500]}")

# Check for tool execution — if deny worked, Bash should NOT have run
raw = json.dumps(output)
bash_ran = '"tool_name": "Bash"' in raw or '"name": "Bash"' in raw
print(f"\nBash tool executed: {bash_ran}")
print(f"Deny worked: {not bash_ran or 'denied' in raw.lower() or 'blocked' in raw.lower()}")

In [ ]:
# Test: error shape — agent should treat as transient failure and retry
print("=" * 60)
print("TEST: error response (transient failure)")
print("=" * 60)

output = run_test(shape_error)

print(f"\nResult type: {type(output).__name__}")
if "result" in output:
    text = output["result"].get("content", [{}])[0].get("text", "(no text)")
    print(f"\nAgent response:\n{text[:500]}")
elif "error" in output:
    print(f"\nError: {output['error'][:500]}")

# Check for retry behavior — errors cause agents to try alternative approaches
raw = json.dumps(output)
retry_signals = ["retry", "try again", "alternative", "different", "instead"]
shows_retry = any(s in raw.lower() for s in retry_signals)
print(f"\nShows retry/alternative behavior: {shows_retry}")

In [ ]:
# Test: systemMessage shape — should NOT block the tool, message is user-visible only
print("=" * 60)
print("TEST: systemMessage alone (no deny)")
print("=" * 60)

shape_system_message_only = {
    "systemMessage": "This is a test system message — does the agent see it?"
}

output = run_test(shape_system_message_only)

print(f"\nResult type: {type(output).__name__}")
if "result" in output:
    text = output["result"].get("content", [{}])[0].get("text", "(no text)")
    print(f"\nAgent response:\n{text[:500]}")
elif "error" in output:
    print(f"\nError: {output['error'][:500]}")

# systemMessage alone should NOT block — tool should proceed
raw = json.dumps(output)
tool_proceeded = "hello" in raw.lower() or "echo" in raw.lower()
print(f"\nTool proceeded (not blocked): {tool_proceeded}")

## Findings Comparison

| Response shape | Expected behavior | Observed |
|---|---|---|
| `{"hookSpecificOutput": {"permissionDecision": "deny", ...}}` | Tool is blocked, agent acknowledges denial | Run test above |
| `{"error": "..."}` | Treated as transient failure, agent retries with variations | Run test above |
| `{"systemMessage": "..."}` alone | User sees message, tool proceeds normally | Run test above |

### Key takeaways (validate against live test output above)

1. **`deny` is the only correct blocking mechanism** — it's read by Claude Code's permission system
2. **`error` causes retry loops** — the agent doesn't interpret errors as policy, just as failures
3. **`systemMessage` is cosmetic** — visible to the user but doesn't affect tool execution or agent behavior

In [ ]:
# make_deny_response() — canonical deny response generator
# Copy this into any PreToolUse hook that needs to block a tool call.
import json

def make_deny_response(reason: str) -> str:
    """Return the JSON string to print to stdout to block a tool call.
    
    Usage in a hook script:
        print(make_deny_response("This path is outside the allowed worktree."))
    """
    response = {
        "hookSpecificOutput": {
            "hookEventName": "PreToolUse",
            "permissionDecision": "deny",
            "permissionDecisionReason": reason
        }
    }
    return json.dumps(response)

# Example
example = make_deny_response("Writing outside the project directory is not permitted.")
print("Example deny response:")
print(example)
print()
print("Paste make_deny_response() into any PreToolUse hook to block tool calls.")

## Codified in CLAUDE.md

This finding is now captured as a project-level rule in `CLAUDE.md`:

> **PreToolUse hooks must use `deny`, not errors**  
> When a PreToolUse hook needs to block an action, return a `deny` response with a reason — not an error. Agents will not interpret error messages as policy; they treat errors as transient failures and retry with variations. A `deny` response is understood as intentional policy and stops retry loops.

See also: the `worktree-isolation` plugin (`plugins/worktree-isolation/hooks/`) for a production example of a PreToolUse hook using the deny pattern.